# 2024 유로 스페인 빌드업 패턴: 왼쪽 쏠림의 이유

지금까지 확인한 "왼쪽 쏠림"(구역 점유·통계 검정·체인 길이)은 전부 "쏠림이 있었다"는 사실만 보여줬습니다. 이 노트북은 그 이유를 세 가지 각도에서 확인합니다(분석 질문 7).

1. **득점/슈팅 효율**: 왼쪽으로 끝난 체인이 오른쪽으로 끝난 체인보다 슈팅·xG·골로 더 자주 이어졌는가?
2. **선수 조합 밀도**: 왼쪽 포지션 슬롯끼리(예: 레프트백-레프트윙)의 패스 연결이 오른쪽보다 더 자주 나타났는가?
3. **크로스/어시스트 위치**: 골 어시스트·슈팅 어시스트·크로스가 왼쪽에서 더 자주 나왔는가?

이 폴더(`chain_outcomes/`)는 "2024 유로 스페인의 빌드업 패턴" 주제의 심화 방법론 전용 하위 폴더입니다. 분석 기획은 [`../PLAN.md`](../PLAN.md), possession 체인 정의는 [`../possession_chains/`](../possession_chains/), 좌우 비대칭 통계는 [`../asymmetry_stats/`](../asymmetry_stats/)를 참고하세요.

## 방법론

1. **체인-슈팅 연결**: `possession_chains`에서 이미 정의한 체인(같은 possession 안에서 이어진 성공 패스 2회 이상, `src.visualizer._prepare_chain_passes`)의 마지막 패스가 도착한 구역(Left/Central/Right, `_chain_length_by_final_side`)별로, 같은 possession id 안에 Shot 이벤트가 있는지 확인해 슈팅 전환율·xG 합계·골 수를 집계합니다.
2. **선수 조합 밀도**: 각 팀 포지션 슬롯(`position` 최빈값)을 좌(L)/우(R)/중앙(C)으로 분류하고, 성공 패스의 (패서 슬롯, 리시버 슬롯) 쌍이 L-L(양쪽 다 왼쪽)인지 R-R(양쪽 다 오른쪽)인지 집계해 매치별 "L-L 비율"(L-L / (L-L+R-R))을 계산합니다.
3. **크로스/어시스트 위치**: StatsBomb의 `pass_goal_assist`(골로 직결)/`pass_shot_assist`(슈팅으로 직결)/`pass_cross`(크로스) 플래그가 참인 패스의 시작 위치(`location`)를 왼쪽/중앙/오른쪽 구역으로 분류해 집계합니다.

세 지표 모두 "왼쪽 비율"(왼쪽 / (왼쪽+오른쪽))에 대해 완전 대칭(0.5)과의 차이를 `asymmetry_stats`와 동일하게 paired t-test로 검정합니다(골 어시스트는 표본이 12개로 너무 작아 검정 없이 원시 개수만 보고합니다).

**데이터 검토** (`scripts/review_chain_outcomes_data.py`, 7경기 전체): Shot 이벤트의 `location`/`shot_statsbomb_xg`/`shot_outcome` 결측 0건. 전체 슈팅 123개 중 111개(90%)가 기존 체인 정의와 같은 possession으로 연결됩니다. `pass_goal_assist`/`pass_shot_assist`/`pass_cross`는 7경기 합산 각각 12/96/89건으로, 크로스·슈팅 어시스트는 검정에 쓸 만하지만 골 어시스트는 표본이 작습니다.

In [ ]:
import os
import sys

if sys.platform.startswith('win') and hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from mplsoccer import Pitch

from src.data_loader import get_competition_matches, get_match_events
from src.visualizer import _prepare_chain_passes, _chain_length_by_final_side, _zone_index

COMPETITION_ID = 55
SEASON_ID = 282
TEAM = "Spain"

pitch = Pitch(pitch_type='statsbomb', positional=True)
x_edges = pitch.dim.positional_x
y_edges = pitch.dim.positional_y
LEFT_Y = {0, 1}
RIGHT_Y = {3, 4}

output_dir = os.path.join(os.getcwd(), "processed")
os.makedirs(output_dir, exist_ok=True)


def match_chain_outcomes(events):
    passes = _prepare_chain_passes(events, TEAM, min_chain_length=2, x_edges=x_edges, y_edges=y_edges)
    summary = _chain_length_by_final_side(passes)

    shots = events[(events['type'] == 'Shot') & (events['team'] == TEAM)]
    shot_agg = shots.groupby('possession').agg(
        xg=('shot_statsbomb_xg', 'sum'),
        n_shots=('shot_statsbomb_xg', 'count'),
        n_goals=('shot_outcome', lambda s: (s == 'Goal').sum()),
    )

    summary = summary.join(shot_agg, on='possession')
    summary[['xg', 'n_shots', 'n_goals']] = summary[['xg', 'n_shots', 'n_goals']].fillna(0)
    summary['has_shot'] = summary['n_shots'] > 0
    return summary


def match_role_pair_density(events):
    team_events = events[events['team'] == TEAM]
    player_position = (
        team_events.dropna(subset=['position'])
        .groupby('player')['position']
        .agg(lambda s: s.value_counts().idxmax())
    )

    passes = events[(events['type'] == 'Pass') & (events['team'] == TEAM)].copy()
    passes = passes[passes['pass_outcome'].isna() & passes['pass_recipient'].notna()]
    passes['passer_position'] = passes['player'].map(player_position)
    passes['recipient_position'] = passes['pass_recipient'].map(player_position)
    passes = passes.dropna(subset=['passer_position', 'recipient_position'])

    def side(pos):
        if pos.startswith('Left'):
            return 'L'
        if pos.startswith('Right'):
            return 'R'
        return 'C'

    passes['passer_side'] = passes['passer_position'].map(side)
    passes['recipient_side'] = passes['recipient_position'].map(side)

    counts = {'L-L': 0, 'R-R': 0, 'Cross': 0, 'Central-involved': 0}
    for _, row in passes.iterrows():
        s1, s2 = row['passer_side'], row['recipient_side']
        if s1 == 'L' and s2 == 'L':
            counts['L-L'] += 1
        elif s1 == 'R' and s2 == 'R':
            counts['R-R'] += 1
        elif s1 == 'C' or s2 == 'C':
            counts['Central-involved'] += 1
        else:
            counts['Cross'] += 1
    return counts


def _side_of_location(loc):
    if not isinstance(loc, list):
        return None
    xi, yi = _zone_index(loc[0], loc[1], x_edges, y_edges)
    if yi in LEFT_Y:
        return 'Left'
    if yi in RIGHT_Y:
        return 'Right'
    return 'Central'


def _bool_col(df, col):
    if col in df.columns:
        return df[col].fillna(False).astype(bool)
    return pd.Series(False, index=df.index)


def match_assist_locations(events):
    passes = events[(events['type'] == 'Pass') & (events['team'] == TEAM)].copy()
    passes['side'] = passes['location'].apply(_side_of_location)

    ga = passes[_bool_col(passes, 'pass_goal_assist')]
    sa = passes[_bool_col(passes, 'pass_shot_assist')]
    cr = passes[_bool_col(passes, 'pass_cross')]

    return {
        'goal_assist_L': (ga['side'] == 'Left').sum(), 'goal_assist_C': (ga['side'] == 'Central').sum(),
        'goal_assist_R': (ga['side'] == 'Right').sum(),
        'shot_assist_L': (sa['side'] == 'Left').sum(), 'shot_assist_C': (sa['side'] == 'Central').sum(),
        'shot_assist_R': (sa['side'] == 'Right').sum(),
        'cross_L': (cr['side'] == 'Left').sum(), 'cross_R': (cr['side'] == 'Right').sum(),
    }

In [ ]:
matches = get_competition_matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
spain_matches = matches[(matches['home_team'] == TEAM) | (matches['away_team'] == TEAM)].copy()
spain_matches = spain_matches.sort_values('match_date')

chain_rows = []
density_rows = []
assist_rows = []
for _, match in spain_matches.iterrows():
    match_id = match['match_id']
    opponent = match['away_team'] if match['home_team'] == TEAM else match['home_team']
    stage = match['competition_stage']

    events = get_match_events(match_id=match_id)

    co = match_chain_outcomes(events)
    for side in ['Left', 'Right', 'Central']:
        side_df = co[co['final_side'] == side]
        chain_rows.append({
            'match': f"{stage} vs {opponent}", 'side': side,
            'n_chains': len(side_df), 'n_chains_with_shot': int(side_df['has_shot'].sum()),
            'xg': round(side_df['xg'].sum(), 3), 'n_goals': int(side_df['n_goals'].sum()),
        })

    density = match_role_pair_density(events)
    density_rows.append({'match': f"{stage} vs {opponent}", **density})

    assist = match_assist_locations(events)
    assist_rows.append({'match': f"{stage} vs {opponent}", **assist})

chain_df = pd.DataFrame(chain_rows)
density_df = pd.DataFrame(density_rows)
density_df['ll_ratio'] = density_df['L-L'] / (density_df['L-L'] + density_df['R-R'])
assist_df = pd.DataFrame(assist_rows)
assist_df['cross_ratio'] = assist_df['cross_L'] / (assist_df['cross_L'] + assist_df['cross_R'])

In [ ]:
agg = chain_df.groupby('side')[['n_chains', 'n_chains_with_shot', 'xg', 'n_goals']].sum()
agg['shot_rate'] = (agg['n_chains_with_shot'] / agg['n_chains']).round(3)
agg['xg_per_chain'] = (agg['xg'] / agg['n_chains']).round(4)
agg

In [ ]:
t_stat, p_value = stats.ttest_1samp(density_df['ll_ratio'], popmean=0.5)
print(f"L-L 비율: mean={density_df['ll_ratio'].mean():.3f}, sd={density_df['ll_ratio'].std():.3f}, "
      f"t(6)={t_stat:.3f}, p={p_value:.4f}")
density_df

In [ ]:
ga_totals = assist_df[['goal_assist_L', 'goal_assist_C', 'goal_assist_R']].sum()
sa_totals = assist_df[['shot_assist_L', 'shot_assist_C', 'shot_assist_R']].sum()
print('골 어시스트 합계:', ga_totals.to_dict())
print('슈팅 어시스트 합계:', sa_totals.to_dict())

sa_sub = assist_df[(assist_df['shot_assist_L'] + assist_df['shot_assist_R']) > 0].copy()
sa_sub['ratio'] = sa_sub['shot_assist_L'] / (sa_sub['shot_assist_L'] + sa_sub['shot_assist_R'])
t_sa, p_sa = stats.ttest_1samp(sa_sub['ratio'], popmean=0.5)
print(f"슈팅 어시스트 L 비율: mean={sa_sub['ratio'].mean():.3f}, n={len(sa_sub)}, t={t_sa:.3f}, p={p_sa:.4f}")

t_cr, p_cr = stats.ttest_1samp(assist_df['cross_ratio'], popmean=0.5)
print(f"크로스 L 비율: mean={assist_df['cross_ratio'].mean():.3f}, t(6)={t_cr:.3f}, p={p_cr:.4f}")
assist_df

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5.5))
fig.set_facecolor('#1e1e1e')

for ax in (ax1, ax2, ax3):
    ax.set_facecolor('#1e1e1e')
    for spine in ax.spines.values():
        spine.set_color('#555555')
    ax.tick_params(colors='white')

side_order = ['Left', 'Central', 'Right']
side_colors = {'Left': '#ffe14d', 'Central': '#00f0ff', 'Right': '#ff5cad'}
x = np.arange(len(side_order))
ax1.bar(x, [agg.loc[s, 'xg_per_chain'] for s in side_order],
        color=[side_colors[s] for s in side_order])
ax1.set_xticks(x)
ax1.set_xticklabels(side_order, color='white')
ax1.set_ylabel('xG per chain', color='white')
ax1.set_title('Chain efficiency by final side', color='white', fontweight='bold')
for i, s in enumerate(side_order):
    ax1.text(i, agg.loc[s, 'xg_per_chain'] + 0.001,
              f"{agg.loc[s, 'shot_rate']*100:.0f}% shot rate\n{int(agg.loc[s, 'n_goals'])} goals",
              ha='center', color='white', fontsize=8)

y_positions = np.arange(len(density_df))
ax2.scatter(density_df['ll_ratio'], y_positions, color='#ffe14d', s=70, zorder=3)
ax2.axvline(0.5, color='white', linestyle='--', linewidth=1, alpha=0.7)
ax2.set_yticks(y_positions)
ax2.set_yticklabels(density_df['match'], color='white', fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel('Left-Left pass ratio among L-L + R-R\n(0.5 = symmetric)', color='white', fontsize=9)
ax2.set_title('Left vs right combination density', color='white', fontweight='bold')

y3 = np.arange(len(assist_df))
ax3.scatter(assist_df['cross_ratio'], y3, color='#00f0ff', s=70, zorder=3)
ax3.axvline(0.5, color='white', linestyle='--', linewidth=1, alpha=0.7)
ax3.set_yticks(y3)
ax3.set_yticklabels(assist_df['match'], color='white', fontsize=8)
ax3.invert_yaxis()
ax3.set_xlabel('Left cross ratio among L + R crosses\n(0.5 = symmetric)', color='white', fontsize=9)
ax3.set_title('Cross origin: left vs right', color='white', fontweight='bold')

fig.tight_layout()
out_path = os.path.join(output_dir, 'spain_euro2024_chain_outcomes.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
plt.show()
print('저장 완료:', out_path)

## 관찰 기록

결과 해석과 결론은 `RESULTS.md`에 문서화했습니다.